In [1]:
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.client import MlflowClient
from mlflow.models.signature import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
)
import subprocess

mlflow.set_tracking_uri("http://localhost:5000")
_ = mlflow.set_experiment("Reproducibility_Drill")

2026/08/29 16:26:22 INFO mlflow.tracking.fluent: Experiment with name 'Reproducibility_Drill' does not exist. Creating a new experiment.


In [2]:
# Fetch the current git commit hash
try:
    git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).strip().decode('utf-8')
except Exception:
    git_commit = "unknown"
print(git_commit)

7feab4960d4add090b6935429e80249215048d5a


In [3]:
df = pd.read_csv("Iris.csv")
print(df.head(5))
if 'Id' in df.columns:
    df = df.drop(columns=['Id'])

X = df.drop(columns=['Species'])
y = df['Species']

   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa


In [4]:
seed = 18
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
params = {"n_estimators": 100, "max_depth": 5, "random_state": seed}

In [5]:
with mlflow.start_run() as run:
    model = RandomForestClassifier(**params) # type: ignore
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, preds)
    precision_macro = precision_score(y_test, preds, average="macro", zero_division=0)
    recall_macro = recall_score(y_test, preds, average="macro", zero_division=0)
    f1_macro = f1_score(y_test, preds, average="macro", zero_division=0)
    loss = log_loss(y_test, proba, labels=model.classes_)

    # Log parameters, metrics, seed, and git_commit tag
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc) # type: ignore
    mlflow.log_metric("precision_macro", precision_macro) # type: ignore
    mlflow.log_metric("recall_macro", recall_macro) # type: ignore
    mlflow.log_metric("f1_macro", f1_macro) # type: ignore
    mlflow.log_metric("log_loss", loss) # type: ignore
    mlflow.log_param("seed", seed)
    mlflow.set_tag("git_commit", git_commit)

    # Log artifact and register model
    signature = infer_signature(X_train, preds)
    mlflow.sklearn.log_model( # type: ignore
        sk_model=model,
        artifact_path="model",
        signature=signature,
        registered_model_name="Iris_RF_Classifier"
    )
    
    print(f"Logged run {run.info.run_id} with commit {git_commit}")
    print(f"accuracy={acc:.4f}  precision_macro={precision_macro:.4f}  "
          f"recall_macro={recall_macro:.4f}  f1_macro={f1_macro:.4f}  log_loss={loss:.4f}")

2026/08/29 16:26:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Successfully registered model 'Iris_RF_Classifier'.
2026/08/29 16:26:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Iris_RF_Classifier, version 1


Created version '1' of model 'Iris_RF_Classifier'.


Logged run 7328e02c7a02428fb32d48ff33c94456 with commit 7feab4960d4add090b6935429e80249215048d5a
accuracy=1.0000  precision_macro=1.0000  recall_macro=1.0000  f1_macro=1.0000  log_loss=0.0396
🏃 View run brawny-squid-212 at: http://localhost:5000/#/experiments/1/runs/7328e02c7a02428fb32d48ff33c94456
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [6]:
client = MlflowClient()
latest_version = client.get_latest_versions("Iris_RF_Classifier", stages=["None"])[0].version

client.transition_model_version_stage(
    name="Iris_RF_Classifier",
    version=latest_version,
    stage="Staging"
)
print(f"Model version {latest_version} successfully transitioned to Staging.")

/tmp/ipykernel_4839/1888395092.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions("Iris_RF_Classifier", stages=["None"])[0].version
/tmp/ipykernel_4839/1888395092.py:4: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


Model version 1 successfully transitioned to Staging.
